In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Knihovna Circuit

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
```
</details>
Qiskit SDK obsahuje knihovnu oblíbených Circuit, které můžeš využít jako stavební bloky ve svých vlastních programech. Použití předem definovaných Circuit šetří čas strávený průzkumem, psaním kódu a laděním. Knihovna zahrnuje oblíbené Circuit z oblasti kvantového výpočtu, Circuit, které je obtížné klasicky simulovat, a Circuit užitečné pro benchmarking kvantového hardwaru.

Tato stránka uvádí různé kategorie Circuit, které knihovna poskytuje. Úplný seznam Circuit najdeš v [API dokumentaci circuit library](https://docs.quantum.ibm.com/api/qiskit/circuit_library).
## Standardní Gate
Knihovna Circuit také obsahuje standardní kvantové Gate. Některé jsou základnější Gate (například `UGate`) a jiné jsou víceQubitové Gate, které obvykle potřebují být sestaveny z jedno- a dvouQubitových Gate. Chceš-li přidat importované Gate do svého Circuit, použij metodu `append`; prvním argumentem je Gate a dalším argumentem je seznam Qubitů, na které se Gate aplikuje.

Například následující buňka kódu vytvoří Circuit s Hadamardovým Gate a multi-controlled-X Gate.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import HGate, MCXGate

mcx_gate = MCXGate(3)
hadamard_gate = HGate()

qc = QuantumCircuit(4)
qc.append(hadamard_gate, [0])
qc.append(mcx_gate, [0, 1, 2, 3])
qc.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/a846a845-7ac5-4c92-b124-d2b90a773ba2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/a846a845-7ac5-4c92-b124-d2b90a773ba2-0.svg)

Viz [Standard gates](https://docs.quantum.ibm.com/api/qiskit/circuit_library#standard-gates) v API dokumentaci circuit library.

<CodeAssistantAdmonition tagLine="Nevíš, jak se tvůj Gate jmenuje? Zkus se zeptat Qiskit Code Assistant." />

## N-lokální Circuit

Tyto Circuit střídají vrstvy jednoqubitových rotačních Gate s vrstvami víceQubitových provazujících Gate.

Tato rodina Circuit je oblíbená ve variačních kvantových algoritmech, protože dokáže produkovat širokou škálu kvantových stavů. Variační algoritmy upravují parametry Gate tak, aby nalezly stavy s určitými vlastnostmi (například stavy, které představují dobré řešení optimalizačního problému). Pro tento účel je mnoho Circuit v knihovně **parametrizovaných**, což znamená, že je můžeš definovat bez pevně daných hodnot.

Následující buňka kódu importuje Circuit `n_local`, ve které jsou provazující Gate dvouQubitové Gate. Tento Circuit prokládá bloky parametrizovaných jednoqubitových Gate s provazujícími bloky dvouQubitových Gate. Následující kód vytvoří tříQubitový Circuit s jednoqubitovými RX-Gate a dvouQubitovými CZ-Gate.

In [2]:
from qiskit.circuit.library import n_local

two_local = n_local(3, "rx", "cz")
two_local.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/3ccaeb1b-03c6-4dfa-9000-e48db2516303-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/3ccaeb1b-03c6-4dfa-9000-e48db2516303-0.svg)

Ze atributu `parameters` získáš objekt podobný seznamu s parametry Circuit.

In [3]:
two_local.parameters

ParameterView([ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), ParameterVectorElement(θ[3]), ParameterVectorElement(θ[4]), ParameterVectorElement(θ[5]), ParameterVectorElement(θ[6]), ParameterVectorElement(θ[7]), ParameterVectorElement(θ[8]), ParameterVectorElement(θ[9]), ParameterVectorElement(θ[10]), ParameterVectorElement(θ[11])])

You can also use this to assign these parameters to real values using a dictionary of the form `{ Parameter: number }`. To demonstrate, the following code cell assigns each parameter in the circuit to `0`.

In [4]:
bound_circuit = two_local.assign_parameters(
    {p: 0 for p in two_local.parameters}
)
bound_circuit.decompose().draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/89227b48-12b2-4b1b-9680-55a7fce88a2b-0.svg" alt="Output of the previous code cell" />

Můžeš to také použít k přiřazení těchto parametrů skutečným hodnotám pomocí slovníku ve tvaru `{ Parameter: number }`. Pro ukázku následující buňka kódu přiřadí každý parametr v Circuit hodnotě `0`.

In [5]:
from qiskit.circuit.library import zz_feature_map

features = [0.2, 0.4, 0.8]
feature_map = zz_feature_map(feature_dimension=len(features))

encoded = feature_map.assign_parameters(features)
encoded.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/cf8b1efc-57b3-4681-8e6a-d5b8406d092d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/89227b48-12b2-4b1b-9680-55a7fce88a2b-0.svg)

Další informace najdeš v části [N-local gates](https://docs.quantum.ibm.com/api/qiskit/circuit_library#n-local-circuits) v API dokumentaci circuit library nebo absolvuj [kurz návrhu variačních algoritmů](/learning/courses/variational-algorithm-design) na IBM Quantum Learning.

## Circuit pro kódování dat

Tyto parametrizované Circuit kódují data na kvantové stavy pro zpracování algoritmy kvantového strojového učení. Mezi Circuit podporované Qiskit patří:
 - Amplitudové kódování, které kóduje každé číslo do amplitudy bázového stavu. Tímto způsobem lze uložit $2^n$ čísel v jednom stavu, ale implementace může být nákladná.
 - Bázové kódování, které kóduje celé číslo $k$ přípravou odpovídajícího bázového stavu $|k\rangle$.
 - Úhlové kódování, které nastavuje každé číslo v datech jako úhel otočení v parametrizovaném Circuit.

Nejlepší přístup závisí na specifikách tvé aplikace. Na současných kvantových počítačích však často používáme Circuit s úhlovým kódováním, jako je `zz_feature_map`.

In [6]:
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp


# Prepare an initial state with a Hadamard on the middle qubit
state = QuantumCircuit(3)
state.h(1)

hamiltonian = SparsePauliOp(["ZZI", "IZZ"])
evolution = PauliEvolutionGate(hamiltonian, time=1)

# Evolve state by appending the evolution gate
state.compose(evolution, inplace=True)

state.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/834794df-86e9-4bea-8efa-5380499e359b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/cf8b1efc-57b3-4681-8e6a-d5b8406d092d-0.svg)

Viz [Data encoding circuits](https://docs.quantum.ibm.com/api/qiskit/circuit_library#data-encoding-circuits) v API dokumentaci circuit library.

## Circuit pro časový vývoj

Tyto Circuit simulují vývoj kvantového stavu v čase. Používej Circuit pro časový vývoj ke zkoumání fyzikálních jevů, jako je přenos tepla nebo fázové přechody v systému. Circuit pro časový vývoj jsou také základním stavebním blokem chemických vlnových funkcí (jako jsou zkušební stavy unitárního clusterového přístupu) a algoritmu QAOA, který používáme pro optimalizační problémy.

In [7]:
from qiskit.circuit.library import quantum_volume

quantum_volume(4).draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/9629a507-8191-409e-b895-fd0833c8fcd7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/834794df-86e9-4bea-8efa-5380499e359b-0.svg)

Přečti si [API dokumentaci `PauliEvolutionGate`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PauliEvolutionGate).

## Circuit pro benchmarking a teorii složitosti

Circuit pro benchmarking nám dávají představu o tom, jak dobře náš hardware skutečně funguje, a Circuit z oblasti teorie složitosti nám pomáhají pochopit, jak obtížné jsou problémy, které chceme řešit.

Například benchmark „quantum volume" měří, jak přesně kvantový počítač provádí určitý typ náhodného kvantového Circuit. Skóre kvantového počítače roste s velikostí Circuit, který dokáže spolehlivě spustit. Zohledňuje to všechny aspekty počítače, včetně počtu Qubitů, věrnosti instrukcí, konektivity Qubitů a softwarového zásobníku Transpileru a následného zpracování výsledků. Více o quantum volume si přečti v původním [článku o quantum volume](https://arxiv.org/abs/1811.12926).

Následující kód ukazuje příklad Circuit quantum volume sestaveného v Qiskit, který běží na čtyřech Qubitech (bloky `unitary` jsou náhodné dvouQubitové Gate).

In [8]:
from qiskit.circuit.library import FullAdderGate
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

adder = FullAdderGate(3)  # Adder of 3-bit numbers

# Create the number A=2
reg_a = QuantumRegister(3, "a")
number_a = QuantumCircuit(reg_a)
number_a.initialize(2)  # Number 2; |010>

# Create the number B=3
reg_b = QuantumRegister(3, "b")
number_b = QuantumCircuit(reg_b)
number_b.initialize(3)  # Number 3; |011>

# Create a circuit to hold everything, including a classical register for
# the result
qregs = [
    QuantumRegister(1, "cin"),
    QuantumRegister(3, "a"),
    QuantumRegister(3, "b"),
    QuantumRegister(1, "cout"),
]
reg_result = ClassicalRegister(3)
circuit = QuantumCircuit(*qregs, reg_result)

# Compose number initializers with the adder. Adder stores the result to
# register B, so we'll measure those qubits.
circuit = (
    circuit.compose(number_a, qubits=reg_a)
    .compose(number_b, qubits=reg_b)
    .compose(adder)
)
circuit.measure(reg_b, reg_result)
circuit.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/77555a5a-a81c-47b8-a9ae-3015d84adcf5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/9629a507-8191-409e-b895-fd0833c8fcd7-0.svg)

Knihovna Circuit také zahrnuje Circuit, o nichž se předpokládá, že je obtížné klasicky simulovat, jako jsou Circuit okamžitých kvantových polynomů (iqp). Tyto Circuit vkládají určité diagonální Gate (v výpočetní bázi) mezi bloky Hadamardových Gate.

Mezi další Circuit patří `grover_operator` pro použití v Groverově algoritmu a Circuit `fourier_checking` pro problém kontroly Fourierovy transformace. Tyto Circuit najdeš v části [Particular quantum circuits](https://docs.quantum.ibm.com/api/qiskit/circuit_library#particular-quantum-circuits) v API dokumentaci circuit library.
## Aritmetické Circuit
Aritmetické operace jsou klasické funkce, jako je sčítání celých čísel a bitové operace. Mohou být užitečné s algoritmy, jako je odhadování amplitud pro finanční aplikace, a v algoritmech jako je HHL algoritmus, který řeší soustavy lineárních rovnic.

Jako příklad zkusíme sečíst dvě tříbitová čísla pomocí Circuit „ripple-carry" k provedení sčítání na místě (`FullAdderGate`). Tato sčítačka sečte dvě čísla (budeme je nazývat „A" a „B") a výsledek zapíše do registru, který obsahoval B. V následujícím příkladu je A=2 a B=3.

In [9]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([circuit]).result()

print(f"Count data:\n {result[0].data.c0.get_int_counts()}")

Count data:
 {5: 1024}


![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/77555a5a-a81c-47b8-a9ae-3015d84adcf5-0.svg)

Simulace Circuit ukazuje, že pro všech `1024` měření (tj. s pravděpodobností `1.0`) výstupem je `5`.